# Tokenization

**Tokenization** is the process of breaking down text into smaller units called **tokens**.

## Why tokenize text?

We need to convert text into a sequence of discrete chunks to be usable as inputs to the transformer.
- **Why not break text into individual characters?**
    - Sequences become very long (roughly 4× longer than with subword tokens). The cost of attention grows quadratically with sequence length, and a fixed context window holds far less text.
    - The LLM needs to encode reasoning between the chunks. Using characters means it would first have to construct words from the characters, in order to infer the meaning, before it can begin reasoning. Breaking the text into already meaningful chunks makes the reasoning a lot less complex.
- **Why not break the text into individual words?**
    - There are too many words. English alone has ~170,000 words in current use, and the set is effectively unbounded once you include inflections, names, numbers, misspellings, new words, other languages and code. Any word not in the vocabulary becomes "unknown".
    - Meaning exists at sub-word level. Words like `walk`, `walked`, `walking`, `walker`, `walkers` & `moonwalk` all share common meaning in the chunk `walk`, so treating them all as completely distinct requires learning this concept over and over again. The effect of `ing` on a word can then also be applied to many other words.
    - There is also a subtler reason: once tokenized, we then need to embed the tokens (coming up in the next section). Embedding looks up a learned vector for each token in a table with one row per vocabulary entry. A huge vocabulary means a huge table (and an equally huge output layer), and rare words appear too few times in training for their vectors to be learned well. Subwords let rare words reuse well-trained pieces.

In practice, tokenizers such as BPE learn their subwords from frequency, not grammar, so splits don't always line up with meaningful pieces: a common word like `walking` is a single token, `moonwalk` is split into `moon` · `walk`, and rarer words get split in less intuitive places (`unbelievably` → `unbelie` · `vably`, as we'll see below).

| Tokenization | Sequence length | Vocabulary                | Unknown words            |
| ------------ | --------------- | ------------------------- | ------------------------ |
| Characters   | Very long       | Tiny                      | No                       |
| Words        | Short           | Huge, open-ended          | Yes                      |
| Subwords     | Medium          | Fixed, chosen (~30k–200k) | No (byte-level fallback) |

### Key Concepts

- A **token** is a chunk of text. 1 token ≈ 0.75 words in English (roughly 4 characters).
- Different tokenizers split text differently.
- Common words are often single tokens: `"hello"` → `[hello]`
- Uncommon words get split into multiple tokens: `"tokenization"` → `[token, ization]`
- Punctuation is often its own token: `","` → `[,]`

---

## Setup: Importing Required Libraries

We will use:

- The [transformers](https://huggingface.co/docs/transformers/en/index) library, from the [Hugging Face](https://huggingface.co/) ecosystem, to access the tokenizers for several common models.
- [tiktoken](https://pypi.org/project/tiktoken/), the tokenizer used by OpenAI's modern models.
- [regex](https://pypi.org/project/regex/), an extended version of Python's built-in `re` module, to split text into pieces.

In [1]:
from collections import Counter
from time import sleep

import regex
import tiktoken
from transformers import AutoTokenizer, PreTrainedTokenizerFast

---

## Part 1: Basic Tokenization

Let's start by seeing how a simple sentence is tokenized.

In [2]:
# Load OpenAI's GPT-4 tokenizer
tokenizer = tiktoken.get_encoding("cl100k_base")  # Used by GPT-4 and GPT-3.5-turbo

# Enter a sentence you wish to tokenize
text = "Artificial intelligence is transforming technology."

# Tokenize the text
tokens = tokenizer.encode(text)

# Decode each token back into a string
token_strings = [tokenizer.decode([token]) for token in tokens]

print(f"Original text: {text}")
print(f"Tokens: {tokens}")
print(f"Token strings: {token_strings}")

Original text: Artificial intelligence is transforming technology.
Tokens: [9470, 16895, 11478, 374, 46890, 5557, 13]
Token strings: ['Art', 'ificial', ' intelligence', ' is', ' transforming', ' technology', '.']


A token's ID tells us nothing about its text. Neighbouring IDs decode to completely unrelated strings:

In [3]:
token_a = 11478
token_b = token_a + 1
token_c = token_a - 1

str_a = tokenizer.decode([token_a])
str_b = tokenizer.decode([token_b])
str_c = tokenizer.decode([token_c])

print("The similar token indices decode to the following strings:")
print(f'    {token_c} → "{str_c}"')
print(f'    {token_a} → "{str_a}"')
print(f'    {token_b} → "{str_b}"')

The similar token indices decode to the following strings:
    11477 → "-day"
    11478 → " intelligence"
    11479 → ".IS"


### How does tokenizing text work?

Tokenizer algorithms vary but commonly consist of the following key steps:

1. **Pre-tokenize**: Split the text into word-like chunks using a regular expression (regex).
2. **Convert to bytes**: Turn each chunk into its UTF-8 bytes, so every chunk starts as a sequence of single-byte tokens.
3. **Apply the merges**: The tokenizer has a *ranked list* of known tokens. Repeatedly find the adjacent pair whose combination has the lowest rank, and merge it into one token, until no adjacent pair is in the list.
4. **Look up the IDs**: Replace each token with its integer ID from the vocabulary (in tiktoken, a token's ID is simply its rank). This list of integers is what the LLM actually receives.

In [4]:
# We will use the ranked list of tokens from tiktoken
R = tokenizer._mergeable_ranks

print(f"R contains {len(R):,} entries: the 256 single bytes, then ~100,000 learned merges.")
print(list(R.items())[:10])
print(list(R.items())[-10:])

R contains 100,256 entries: the 256 single bytes, then ~100,000 learned merges.
[(b'!', 0), (b'"', 1), (b'#', 2), (b'$', 3), (b'%', 4), (b'&', 5), (b"'", 6), (b'(', 7), (b')', 8), (b'*', 9)]
[(b' r\xc3\xa9sultats', 100246), (b' moden', 100247), (b' Icelandic', 100248), (b';d', 100249), (b'.allowed', 100250), (b'(newUser', 100251), (b' merciless', 100252), (b'.WaitFor', 100253), (b' daycare', 100254), (b' Conveyor', 100255)]


In [5]:
# Here we tokenize the word ' unbelievably' by first splitting it into single UTF-8 byte parts (characters, in this case)
# and then repeatedly merging together the lowest rank pair of parts according to the ranked list of tokens above.

word = " unbelievably"

# split into single byte parts (chars)
parts = [bytes([b]) for b in word.encode("utf-8")]
print("start:", " | ".join(p.decode() for p in parts))

while True:
    sleep(1.5)  # just to slow things down a bit
    best = None

    # find lowest rank pair and merge
    for i in range(len(parts) - 1):
        r = R.get(parts[i] + parts[i + 1])
        if r is not None and (best is None or r < best[0]):
            best = (r, i)
    if best is None:
        break

    r, i = best
    a, b = parts[i], parts[i + 1]
    parts = parts[:i] + [a + b] + parts[i + 2:]
    print(f"merge {a.decode()!r}+{b.decode()!r} (rank {r}) ->", " | ".join(p.decode() for p in parts))

start:   | u | n | b | e | l | i | e | v | a | b | l | y
merge 'e'+'l' (rank 301) ->   | u | n | b | el | i | e | v | a | b | l | y
merge 'u'+'n' (rank 359) ->   | un | b | el | i | e | v | a | b | l | y
merge 'a'+'b' (rank 370) ->   | un | b | el | i | e | v | ab | l | y
merge 'l'+'y' (rank 398) ->   | un | b | el | i | e | v | ab | ly
merge 'i'+'e' (rank 648) ->   | un | b | el | ie | v | ab | ly
merge ' '+'un' (rank 653) ->  un | b | el | ie | v | ab | ly
merge 'ab'+'ly' (rank 2915) ->  un | b | el | ie | v | ably
merge 'b'+'el' (rank 9978) ->  un | bel | ie | v | ably
merge 'bel'+'ie' (rank 32898) ->  un | belie | v | ably
merge ' un'+'belie' (rank 40037) ->  unbelie | v | ably
merge 'v'+'ably' (rank 89234) ->  unbelie | vably


### Tokenizing our example text

Let's tokenize our example text: "Artificial intelligence is transforming technology."

In [6]:
# First, define the merging process above as a function
def bpe(piece, ranks):
    """Split a chunk of bytes into tokens by repeatedly merging the lowest-rank pair."""
    parts = [bytes([b]) for b in piece]

    while True:
        best = None

        # find lowest rank pair and merge
        for i in range(len(parts) - 1):
            r = ranks.get(parts[i] + parts[i + 1])
            if r is not None and (best is None or r < best[0]):
                best = (r, i)
        if best is None:
            break

        r, i = best
        parts = parts[:i] + [parts[i] + parts[i + 1]] + parts[i + 2:]

    return parts


# A small subset of tiktoken's ranked list, containing just the tokens needed for our example text
demo_ranks = {
    # Merges used to build "Artificial" (it isn't a single token)
    b"al": 278,
    b"ic": 292,
    b"if": 333,
    b"ial": 532,
    b"ific": 1104,
    b"rt": 3423,
    b"Art": 9470,
    b"ificial": 16895,
    # The other pieces are already whole tokens
    b" intelligence": 11478,
    b" is": 374,
    b" transforming": 46890,
    b" technology": 5557,
    b".": 13,
}

# Reverse lookup, from token ID back to bytes
rank_lookup = {value: key for key, value in demo_ranks.items()}

In [7]:
# Use a regex to split the text into pieces (this is GPT-4's pre-tokenization pattern)
pattern = r"'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s"

pieces = regex.findall(pattern, text)
pieces

['Artificial', ' intelligence', ' is', ' transforming', ' technology', '.']

In [8]:
# Tokenize each piece as we did for the word ' unbelievably'
def encode(text, ranks):
    """Split text into pieces with the regex, then BPE each piece into token IDs."""
    tokens = []
    for piece in regex.findall(pattern, text):
        piece = piece.encode("utf-8")

        if piece in ranks:  # fast path: the whole piece is already a token
            tokens.append(ranks[piece])
        else:  # slow path: merge the piece's bytes, lowest rank first
            parts = bpe(piece, ranks)
            tokens.extend(ranks[part] for part in parts)
    return tokens


tokens = encode(text, demo_ranks)

# These are the numerical tokens produced, and the text chunks they represent
print(f"Tokens: {tokens}")

for token in tokens:
    print(rank_lookup[token], ":", token)

Tokens: [9470, 16895, 11478, 374, 46890, 5557, 13]
b'Art' : 9470
b'ificial' : 16895
b' intelligence' : 11478
b' is' : 374
b' transforming' : 46890
b' technology' : 5557
b'.' : 13


In [9]:
# This gives us the same as the output of the tiktoken tokenizer
print("ours:    ", tokens)
print("tiktoken:", tokenizer.encode(text))
print("decoded: ", [rank_lookup[token] for token in tokens])

ours:     [9470, 16895, 11478, 374, 46890, 5557, 13]
tiktoken: [9470, 16895, 11478, 374, 46890, 5557, 13]
decoded:  [b'Art', b'ificial', b' intelligence', b' is', b' transforming', b' technology', b'.']


### Observations

Notice how:
- Each token's ID is just a number; it indicates nothing about the content of the string.
- Each word is typically one token.
- Longer words are split into multiple tokens.
- Punctuation marks are often separate tokens.
- Spaces can be included with some tokens.
- **All we need to tokenize any text is a regex and a ranked list of tokens.**

### Note

Anything that can be encoded as UTF-8 can be tokenized in this way, even complex characters and symbols. Rare characters are split into several byte-level tokens:

In [10]:
for symbol in "中𓀀🫠":
    encoded = symbol.encode("utf-8")
    tokenized = tokenizer.encode(symbol)
    print(symbol, ":", encoded, ":", tokenized)

中 : b'\xe4\xb8\xad' : [16325]
𓀀 : b'\xf0\x93\x80\x80' : [172, 241, 222, 222]
🫠 : b'\xf0\x9f\xab\xa0' : [9468, 104, 254]


In [11]:
# Decode each of the emoji's tokens on its own
tokenized = tokenizer.encode("🫠")

for t in tokenized:
    b = tokenizer.decode_single_token_bytes(t)
    try:
        print(t, ":", b.decode("utf-8"))
    except UnicodeDecodeError:
        print(t, ":", b, "(part of a character)")

# Only once all the bytes are joined back together do they form a valid character
print(tokenized, "→", tokenizer.decode(tokenized))

9468 : b'\xf0\x9f' (part of a character)
104 : b'\xab' (part of a character)
254 : b'\xa0' (part of a character)
[9468, 104, 254] → 🫠


---

## Part 2: Learning the Ranked List

If all we need to tokenize text is a ranked list of tokens, which we borrowed from tiktoken, where did tiktoken get it from?

It's learned from text, in a very similar process to how it is applied to our text:

1. **Start with bytes**: The vocabulary begins as the 256 possible byte values, so any text can already be represented.
2. **Pre-tokenize**: A regex splits the training text into word-like chunks (words with their leading space, punctuation, groups of up to 3 digits). Merges never cross these boundaries.
3. **Count pairs**: Count how often each pair of adjacent tokens appears across all the chunks.
4. **Merge the most frequent pair**: Add it to the vocabulary as a new token, with the next rank, and replace every occurrence of the pair in the text.
5. **Repeat**: Repeat steps 3–4 until the vocabulary reaches its target size (e.g. ~100k tokens for GPT-4).

The result is the regex plus a ranked list of tokens. A lower rank means the token was learned earlier because it was more frequent. Encoding later replays those merges in rank order, which is why `in`, `er` and `on` merge first and whole words like ` international` come later.

In [12]:
# Viewing the start and end of tiktoken's ranked list again, this makes sense
print(list(R.items())[256:266])  # Most common patterns (after the base 256)
print(list(R.items())[-10:])  # Less common patterns

[(b'  ', 256), (b'    ', 257), (b'in', 258), (b' t', 259), (b'        ', 260), (b'er', 261), (b'   ', 262), (b'on', 263), (b' a', 264), (b're', 265)]
[(b' r\xc3\xa9sultats', 100246), (b' moden', 100247), (b' Icelandic', 100248), (b';d', 100249), (b'.allowed', 100250), (b'(newUser', 100251), (b' merciless', 100252), (b'.WaitFor', 100253), (b' daycare', 100254), (b' Conveyor', 100255)]


In [13]:
# Training our own ranked token list on our own text
def train_ranks(text, vocab_size):
    """Learn a ranked vocabulary of byte strings from text."""
    # 1. start with all 256 single bytes
    ranks = {bytes([i]): i for i in range(256)}

    # 2. pre-tokenize, and count how often each chunk occurs
    chunk_counts = Counter(regex.findall(pattern, text))
    chunks = {tuple(bytes([b]) for b in c.encode("utf-8")): n for c, n in chunk_counts.items()}

    while len(ranks) < vocab_size:
        # 3. count adjacent pairs across all chunks
        pair_counts = Counter()
        for parts, n in chunks.items():
            for a, b in zip(parts, parts[1:]):
                pair_counts[a, b] += n
        if not pair_counts:
            break

        # 4. add the most frequent pair to the vocabulary with the next rank
        (a, b), _ = pair_counts.most_common(1)[0]
        ranks[a + b] = len(ranks)

        # merge that pair everywhere it occurs
        new_chunks = {}
        for parts, n in chunks.items():
            merged, i = [], 0
            while i < len(parts):
                if i < len(parts) - 1 and parts[i] == a and parts[i + 1] == b:
                    merged.append(a + b)
                    i += 2
                else:
                    merged.append(parts[i])
                    i += 1
            new_chunks[tuple(merged)] = new_chunks.get(tuple(merged), 0) + n
        chunks = new_chunks

    return ranks

In [14]:
with open("assets/blue_lyrics.txt") as f:
    lyrics = f.read()

lyrical_ranks = train_ranks(lyrics, vocab_size=256 + 200)
lyric_lookup = {value: key for key, value in lyrical_ranks.items()}

# Show the 200 learned tokens (skipping the 256 single bytes), in the order they were learned
print([token for token, rank in lyrical_ranks.items() if rank >= 256])

[b' d', b' b', b' ba', b'ee', b' da', b'Da', b' dee', b' di', b' a', b'lu', b'lue', b'nd', b' t', b'he', b' blue', b' i', b're', b'ha', b've', b'Blue', b'ou', b' w', b' the', b'li', b' s', b' are', b' li', b'hat', b'And', b'or', b'hi', b' and', b'ts', b' in', b' al', b' all', b' is', b'id', b'ide', b"'m", b' I', b'te', b' he', b'That', b'se', b' c', b'le', b' g', b'hin', b' hi', b' to', b' ha', b' have', b'ree', b' m', b' here', b' st', b' live', b' wor', b' e', b' eve', b' ever', b' every', b'ke', b' him', b' ins', b' inside', b' ou', b' outs', b' outside', b' h', b' hou', b' house', b'Wi', b'Wit', b'With', b' wi', b' wind', b' windo', b' window', b' cor', b' corve', b' corvet', b' corvette', b' f', b'lf', b' ar', b' arou', b' around', b' lis', b' liste', b' listen', b' lit', b' litt', b' little', b' n', b' everyt', b' everythin', b' everything', b'bo', b'bod', b'body', b' co', b' col', b' colou', b' colour', b' o', b' of', b' that', b' we', b' wea', b' wear', b' stree', b' streets', 

In [15]:
# Tokenize a new sentence using our ranked list learned from the lyrics
alt_text = "I have a blue corvette outside the window of my house"

print(regex.findall(pattern, alt_text))

lyric_tokens = encode(alt_text, lyrical_ranks)

# These are the numerical tokens produced, and the text chunks they represent
print(f"Tokens: {lyric_tokens}")

for token in lyric_tokens:
    print(lyric_lookup[token], ":", token)

['I', ' have', ' a', ' blue', ' corvette', ' outside', ' the', ' window', ' of', ' my', ' house']
Tokens: [73, 308, 264, 270, 339, 325, 278, 335, 363, 391, 328]
b'I' : 73
b' have' : 308
b' a' : 264
b' blue' : 270
b' corvette' : 339
b' outside' : 325
b' the' : 278
b' window' : 335
b' of' : 363
b' my' : 391
b' house' : 328


---

## Part 3: Comparing Different Tokenizers

Different models use different tokenizers. Let's see how our example text, "Artificial intelligence is transforming technology.", is tokenized differently.

In [16]:
# transformers.AutoTokenizer gives us access to a bunch of tokenizers shared on the
# Hugging Face platform by name. We will load 4 examples:
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
t5_tok = AutoTokenizer.from_pretrained("t5-small")
# AutoTokenizer loads this one incorrectly (it drops the spaces), so we load it directly
deep_tok = PreTrainedTokenizerFast.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B")

# Tokenize our sample text with each
special = True
gpt2_tokens = gpt2_tok.encode(text, add_special_tokens=special)
bert_tokens = bert_tok.encode(text, add_special_tokens=special)
t5_tokens = t5_tok.encode(text, add_special_tokens=special)
deep_tokens = deep_tok.encode(text, add_special_tokens=special)

print(f'Sample text: "{text}"')
print()
print(f"gpt2 tokens:\t{gpt2_tokens}")
print(f"bert tokens:\t{bert_tokens}")
print(f"t5 tokens:\t{t5_tokens}")
print(f"deepseek tokens:\t{deep_tokens}")

Sample text: "Artificial intelligence is transforming technology."

gpt2 tokens:	[8001, 9542, 4430, 318, 25449, 3037, 13]
bert tokens:	[101, 7976, 4454, 2003, 17903, 2974, 1012, 102]
t5 tokens:	[24714, 6123, 19, 3, 21139, 748, 5, 1]
deepseek tokens:	[9470, 16895, 11478, 374, 46890, 5557, 13]


We can reverse the process and decode the tokens from their numerical values back to natural language, provided we use the right tokenizer. The wrong tokenizer will not work:

In [17]:
print("Correct tokenizer:")
print(gpt2_tok.decode(gpt2_tokens))
print()
print("Incorrect tokenizer:")
print(bert_tok.decode(gpt2_tokens))

Correct tokenizer:
Artificial intelligence is transforming technology.

Incorrect tokenizer:
revised luxuryah [unused313] truncated interest [unused12]


---

## Part 4: Why Tokenization Matters

The LLM never sees letters or digits, only token IDs. This explains some of the odd mistakes LLMs make.

In [18]:
def show_tokens(text):
    """Print the GPT-4 tokens for a piece of text."""
    ids = tokenizer.encode(text)
    print(f"{text!r:28} → {' · '.join(repr(tokenizer.decode([i])) for i in ids)}   {ids}")


# 1. Spelling: how many r's are in strawberry?
show_tokens("How many r's are in strawberry?")
show_tokens("strawberry")

"How many r's are in strawberry?" → 'How' · ' many' · ' r' · "'s" · ' are' · ' in' · ' strawberry' · '?'   [4438, 1690, 436, 596, 527, 304, 73700, 30]
'strawberry'                 → 'str' · 'aw' · 'berry'   [496, 675, 15717]


**Spelling and letter counting:** In a sentence, ` strawberry` is a *single* token. The model receives one number, with no direct view of the letters inside it, so questions like "how many r's are in strawberry?" are surprisingly hard. It has to have memorised the spelling of each token.

In [19]:
# 2. Arithmetic: how are numbers split?
show_tokens("1234567")
show_tokens("12345 + 67890 = 80235")

'1234567'                    → '123' · '456' · '7'   [4513, 10961, 22]
'12345 + 67890 = 80235'      → '123' · '45' · ' +' · ' ' · '678' · '90' · ' =' · ' ' · '802' · '35'   [4513, 1774, 489, 220, 17458, 1954, 284, 220, 13135, 1758]


**Arithmetic:** Numbers are cut into chunks of up to 3 digits, counted from the left. In `12345` the chunks are `123` · `45`, so the chunks don't line up with units, tens and hundreds, and the same digit can land in a different chunk depending on how long the number is. This makes column-by-column arithmetic harder to learn.

In [20]:
# 3. Capitals and spaces: are these the same word?
for word in ["listen", " listen", " Listen", " LISTEN"]:
    show_tokens(word)

'listen'                     → 'listen'   [37437]
' listen'                    → ' listen'   [9020]
' Listen'                    → ' Listen'   [33249]
' LISTEN'                    → ' LIST' · 'EN'   [28064, 965]


**Capitals and spaces:** To us these are the same word, but to the model they are unrelated token IDs (or, for ` LISTEN`, two tokens). The model has to learn separately that they mean the same thing, which is why unusual capitalization or spacing can change how well a prompt works.

---

## Part 5: Hands-On Exercises

Try these exercises to deepen your understanding.

### Exercise 1: Decode the Individual Tokens

**Task:** Instead of decoding all the tokens at once to reproduce the text, decode each token one at a time so we can see the individual tokens.

**To think about:**
1. Do all tokenizers work the same way?
2. How is punctuation handled differently?

**Hint:** We can see the GPT-2 tokens using:
```python
for token in gpt2_tokens:
    token_string = gpt2_tok.decode([token])
    print(f"{token}\t -> \t'{token_string}'")
```

Try decoding the tokens from the other tokenizers we used above.

In [21]:
# Exercise 1 Answers
# using t5
for token in t5_tokens:
    token_string = t5_tok.decode([token])
    print(f"{token}\t -> \t'{token_string}'")

24714	 -> 	'Artificial'
6123	 -> 	'intelligence'
19	 -> 	'is'
3	 -> 	''
21139	 -> 	'transforming'
748	 -> 	'technology'
5	 -> 	'.'
1	 -> 	'</s>'


In [22]:
# using deepseek
for token in deep_tokens:
    token_string = deep_tok.decode([token])
    print(f"{token}\t -> \t'{token_string}'")

9470	 -> 	'Art'
16895	 -> 	'ificial'
11478	 -> 	' intelligence'
374	 -> 	' is'
46890	 -> 	' transforming'
5557	 -> 	' technology'
13	 -> 	'.'


### Exercise 2: Analyze Your Own Text

**Task:** Take a paragraph from your research, documentation, or a recent email and analyze its tokenization.

**Questions to answer:**
1. How many tokens does it use?
2. What's the character-to-token ratio?
3. Which words get split into multiple tokens?
4. How much would it cost to process this text 1000 times with GPT-4, given `gpt4_input_price_per_1k_tokens_usd = 0.03`?

    *This is the original GPT-4 price, used here as an example. Current prices for OpenAI's models are listed on the [OpenAI API pricing page](https://openai.com/api/pricing/).*

In [ ]:
# Exercise 2 Answers: Analyze your own text (worked example)
my_text = (
    "In our LLM workshop, we compare tokenization strategies to understand cost, "
    "latency, and context-window limits. A single sentence can split into very "
    "different token sequences depending on punctuation, capitalization, and "
    "compound terms such as fine-tuning and retrieval-augmented generation."
)

# 1. How many tokens does it use?
token_ids = tokenizer.encode(my_text)
token_count = len(token_ids)
print(f"1) Token count: {token_count}")

# 2. What is the character-to-token ratio for your chosen tokenizer?
char_count = len(my_text)
char_to_token_ratio = char_count / token_count if token_count else 0
print(f"2) Character-to-token ratio: {char_to_token_ratio:.2f} ({char_count} chars / {token_count} tokens)")

# 3. Which words get split into multiple tokens?
words = [w.strip(".,!?;:\"'()[]{}") for w in my_text.split()]
split_words = []
for w in words:
    if not w:
        continue
    w_tokens = tokenizer.encode(w)
    if len(w_tokens) > 1:
        split_words.append((w, len(w_tokens), w_tokens))

print("3) Words split into multiple tokens:")
if split_words:
    for word, n_tokens, ids in split_words:
        print(f"   - {word}: {n_tokens} tokens -> {ids}")
else:
    print("   - None")

# 4. How much would it cost to process this text 1000 times with GPT-4?
gpt4_input_price_per_1k_tokens_usd = 0.03
cost_per_run_usd = (token_count / 1000) * gpt4_input_price_per_1k_tokens_usd
cost_1000_runs_usd = cost_per_run_usd * 1000

print(
    f"4) Cost for 1000 runs with GPT-4 input pricing (${gpt4_input_price_per_1k_tokens_usd}/1K tokens): "
    f"${cost_1000_runs_usd:.4f}"
)

1) Token count: 54
2) Character-to-token ratio: 5.41 (292 chars / 54 tokens)
3) Words split into multiple tokens:
   - LLM: 2 tokens -> [4178, 44]
   - workshop: 2 tokens -> [1816, 8845]
   - tokenization: 2 tokens -> [5963, 2065]
   - strategies: 2 tokens -> [496, 70488]
   - understand: 2 tokens -> [8154, 2752]
   - latency: 2 tokens -> [5641, 2301]
   - context-window: 2 tokens -> [2196, 42866]
   - punctuation: 2 tokens -> [79, 73399]
   - capitalization: 2 tokens -> [66163, 2065]
   - fine-tuning: 3 tokens -> [63157, 2442, 38302]
   - retrieval-augmented: 6 tokens -> [265, 9104, 838, 7561, 773, 28078]
4) Cost for 1000 runs with GPT-4 input pricing ($0.03/1K tokens): $1.6200


### Exercise 3: Optimize a Prompt

**Task:** You have this verbose prompt. Can you reduce the token count while keeping the same meaning?

Original prompt:
```
"Please provide me with a comprehensive and detailed explanation regarding
the various different ways in which machine learning algorithms can be
utilized and applied in the field of healthcare and medical diagnostics."
```

**Goal:** Reduce tokens by at least 30% without losing meaning.

In [24]:
# Exercise 3 Answer
original_prompt = """Please provide me with a comprehensive and detailed explanation
regarding the various different ways in which machine learning algorithms can be
utilized and applied in the field of healthcare and medical diagnostics."""

optimized_prompt = """Explain in detail how machine learning is used in healthcare and medical diagnostics."""

# Compare token counts
original_tokens = len(tokenizer.encode(original_prompt))
optimized_tokens = len(tokenizer.encode(optimized_prompt))

print(f"Original: {original_tokens} tokens")
print(f"Optimized: {optimized_tokens} tokens")
print(f"Reduction: {((original_tokens - optimized_tokens) / original_tokens * 100):.1f}%")

Original: 37 tokens
Optimized: 15 tokens
Reduction: 59.5%


---

## 🎓 Key Takeaways

By now, you should understand:

- **What tokenization is** and why LLMs need it
- **How BPE tokenizers work**: a regex and a ranked list of merges, learned from text
- **Why different tokenizers produce different results** for the same text
- **Token counts directly impact costs**: optimize your prompts!
- **Context windows are measured in tokens**: manage them carefully

### Practical Applications

1. **Cost Optimization:** Shorter, clearer prompts = lower costs
2. **Multilingual Apps:** Budget more for non-English languages
3. **Context Management:** Track tokens to avoid hitting limits

### 📚 Further Exploration

- [OpenAI Tokenizer Tool](https://platform.openai.com/tokenizer) - Interactive web tool
- [tiktoken GitHub](https://github.com/openai/tiktoken) - Source code and documentation
- [Hugging Face Tokenizers](https://huggingface.co/docs/tokenizers/) - Learn about other tokenizers

## What doesn't Tokekinzation achieve?

In [71]:
# Unlike the numbers used in classic Neural Networks, tokens are still not ordinally meaningful. 

# Token 9470 has nothing in common with 9471
print("Token #9470: ", tokenizer.decode([9470]))
print("Token #9471: ", tokenizer.decode([9471]))

# Their token numbers are similar only because they occurred with similar frequency in the training text. 

Token #9470:  Art
Token #9471:  night


In [78]:
# Mathematics with token numbers is still meaningless. 
fish_token = tokenizer.encode("fish")[0]
do_token = tokenizer.encode("do")[0]

result_token = fish_token+do_token
print(f"{fish_token} + {do_token} = {result_token}")
print(f"fish + do = {tokenizer.decode([result_token])}")

# we might want "fishing" to be the result of adding do to fish. 
# But this would be asking a lot... 

18668 + 3055 = 21723
fish + do =  firstName
